# Act 5: skills on the shared planner

Reuse the Act 4 planner and add a named **Skills** layer. Compare a direct stock lookup, a tasting task that loads relevant guidance, and a sourcing task that needs a template plus human review.

Start with repeatable **offline fixtures**. This notebook requires Python 3.13+; Colab runtime compatibility has not been validated here. No purchase or email transmission occurs. An approved outreach action writes local HTML only.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

## 1. A simple question needs no skill body

**Context engineering:** the model initially sees skill metadata, not every skill's instructions and assets. This fixture answers the stock question directly through a tool.


In [ ]:
from pathlib import Path
from acts.act5_skills import build_act5
from formaggio.agents.skill_layer import Skills
from formaggio.operations.observability import Recorder
from cli import print_harness_result

DB_PATH = Path("outputs/act5_notebook.sqlite")
OUTPUT_ROOT = Path("outputs/act5_notebook")


In [ ]:
stock_lesson = build_act5(fixture=True, scenario="stock-question", output_root=OUTPUT_ROOT)
with Recorder(DB_PATH) as recorder:
    stock_result = await stock_lesson.run(recorder, progress=print)
print_harness_result(stock_result)
print("Skill bodies loaded:", stock_result["skills_loaded"])


## 2. Load relevant guidance for a tasting plan

**Skills:** this fixture loads the tasting-planning body, serving guide, and plan outline. It does not load vendor-outreach instructions. The SDK performs the reads; the harness checks the declared paths and unchanged file contents.


In [ ]:
tasting_lesson = build_act5(fixture=True, scenario="tasting-plan", output_root=OUTPUT_ROOT)
with Recorder(DB_PATH) as recorder:
    tasting_result = await tasting_lesson.run(recorder, progress=print)
print_harness_result(tasting_result)
print("Resources read:", tasting_result["skill_resources"])


## 3. Inspect the named layer

`build_act5` uses the same planner and runtime as Act 4, then adds `Skills(skills_root)`. Each `.run(...)` creates a fresh file snapshot and loading state. Editing a skill between runs updates the next snapshot; editing a file during a run blocks access to that changed file.


In [ ]:
import inspect
print(inspect.getsource(build_act5))
print(inspect.getsource(Skills))


## 4. A template does not grant approval

**Agent harness:** the sourcing fixture loads the approved template before drafting. The host callback then declines the exact draft. Skills guide content and procedure; recipient policy and human review govern the action.


In [ ]:
async def fixture_review(review):
    print(f"Review: {review.draft.recipient} — {review.draft.subject}")
    print(review.draft.body)
    return "decline"  # An explicit host decision. Change to approve for a local HTML save.

outreach_lesson = build_act5(fixture=True, output_root=OUTPUT_ROOT, decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    outreach_result = await outreach_lesson.run(recorder, reviewer=fixture_review, progress=print)
print_harness_result(outreach_result)


## 5. Remove skills and observe the dependency

Removing the layer removes metadata and loading tools. It cannot remove Act 5's template requirement. The sourcing task reports a blocked draft because no approved template was loaded; it never asks the host to approve a nonexistent draft. Direct stock lookup remains available.


In [ ]:
with Recorder(DB_PATH) as recorder:
    without_skills_result = await outreach_lesson.without("skills").run(
        recorder, reviewer=fixture_review, progress=print)
print_harness_result(without_skills_result)


## 6. Try live skill selection (optional)

The cells above use prescribed tool choices to make the comparison repeatable. These cells use a live model, so you can observe its own selection of relevant skills. Confirm the recipient and draft body before approving a local save.


In [ ]:
import os
from getpass import getpass
from formaggio.config import MODEL

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


In [ ]:
async def review_email(review):
    print(f"Recipient: {review.draft.recipient}\nSubject: {review.draft.subject}\n{review.draft.body}")
    while True:
        decision = input("Approve local HTML save or decline? ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision

with Recorder(DB_PATH) as recorder:
    live_result = await build_act5(model=MODEL, output_root=OUTPUT_ROOT).run(
        recorder, reviewer=review_email, progress=print)
print_harness_result(live_result)


## Reflect

- Which information appeared initially, and which arrived through skill tools?
- Why can a stock question succeed without loading a skill body?
- Why does removing skills block this sourcing task rather than bypass the template check?
- Which state is fresh on every invocation, and which preference memory can persist?
- **Evaluation:** compare selected skills, loaded resources, policy events, review decisions, and actual artifacts. Loading a skill proves access, not correct use of its guidance.

Act 6 composes these ideas with the ordering workflow.
